# Personal Expense Tracker

**Lumexa Python Foundations · Course 2: Problem Solving & Program Design · Lessons 6–7**

A program that stores, retrieves, validates and organizes financial records in a file.

* **Part 1 · Lesson 6:** input validation and error handling with `try` and `except`.
* **Part 2 · Lesson 7:** saving the expenses to a JSON file and exporting them to CSV.

▶ Run the code cell below (**Shift + Enter**) and type your answers when the program asks. To keep your own copy, choose **File → Save a copy in Drive**. The program saves `expenses.json` and `expenses.csv` in the files of this Colab session.

In [ ]:
# ======================================================================
#  PERSONAL EXPENSE TRACKER
#  Lumexa Python Foundations · Course 2: Problem Solving & Program Design
#  Part 1 · Lesson 6: error handling and input validation
#  Part 2 · Lesson 7: working with files and structured data (JSON and CSV)
# ======================================================================
import csv
import json
from pathlib import Path


# ----------------------------------------------------------------------
# PART 1 · Lesson 6: never trust what the user types
# ----------------------------------------------------------------------
CATEGORIES = ["food", "transport", "school", "fun", "other"]


def read_amount(text):
    """Turn text into a positive amount, or raise a ValueError that says what is wrong."""
    try:
        amount = float(text)
    except ValueError:
        raise ValueError("'" + text.strip() + "' is not a number.")
    if amount <= 0:
        raise ValueError("The amount must be more than 0.")
    if amount > 100000:
        raise ValueError("That amount looks far too big. Please check it.")
    return amount


def read_date(text):
    """A date must look like YYYY-MM-DD, for example 2030-07-14."""
    parts = text.strip().split("-")
    if len(parts) != 3:
        raise ValueError("Use the form YYYY-MM-DD, for example 2030-07-14.")
    year, month, day = parts
    if not (year.isdigit() and month.isdigit() and day.isdigit()) or len(year) != 4:
        raise ValueError("A date contains only numbers and dashes: YYYY-MM-DD.")
    if not 1 <= int(month) <= 12:
        raise ValueError("The month must be from 1 to 12.")
    if not 1 <= int(day) <= 31:
        raise ValueError("The day must be from 1 to 31.")
    return year + "-" + month.zfill(2) + "-" + day.zfill(2)


def read_category(text):
    """The category must be one of the known ones (capital letters and spaces do not matter)."""
    category = text.strip().lower()
    if category not in CATEGORIES:
        raise ValueError("Choose one of: " + ", ".join(CATEGORIES) + ".")
    return category


def ask_until_valid(prompt, reader):
    """Keep asking until `reader` accepts the answer. try / except / else decide what happens."""
    while True:
        text = input(prompt)
        try:
            value = reader(text)
        except ValueError as problem:
            print("⚠️ ", problem)
        else:
            return value                    # no error: the answer is good


def add_expense(expenses):
    """Ask for every detail (each one is validated) and add the record to the list."""
    date = ask_until_valid("Date (YYYY-MM-DD): ", read_date)
    category = ask_until_valid("Category (" + "/".join(CATEGORIES) + "): ", read_category)
    description = input("What was it for? ").strip()
    amount = ask_until_valid("Amount: ", read_amount)
    expense = {"date": date, "category": category, "description": description, "amount": amount}
    expenses.append(expense)
    return expense


def show_expenses(expenses, empty_message="(no expenses yet)"):
    if len(expenses) == 0:
        print("   " + empty_message)
        return
    number = 1
    total = 0
    for expense in expenses:
        print(f"{number:>3}. {expense['date']}  {expense['category']:<10}{expense['description']:<22}{expense['amount']:>9.2f}")
        total += expense["amount"]
        number += 1
    print(f"     Total: {total:.2f}")


def get_total(pair):
    return pair[1]


def show_totals_by_category(expenses):
    totals = {}
    for expense in expenses:
        totals[expense["category"]] = totals.get(expense["category"], 0) + expense["amount"]
    if len(totals) == 0:
        print("   (no expenses yet)")
    for category, total in sorted(totals.items(), key=get_total, reverse=True):      # largest first
        print(f"   {category:<10}{total:>9.2f}")


# ----------------------------------------------------------------------
# PART 2 · Lesson 7: the expenses are kept in files, so they survive between runs
# ----------------------------------------------------------------------
DATA_FILE = Path("expenses.json")
CSV_FILE = Path("expenses.csv")


def load_expenses():
    """Read the saved expenses. A missing or damaged file must never crash the program."""
    try:
        with open(DATA_FILE, "r", encoding="utf-8") as file:
            data = json.load(file)
    except FileNotFoundError:
        print("No saved file yet, so we start with an empty list.")
        return []
    except json.JSONDecodeError:
        print("The saved file is damaged, so we start with an empty list.")
        return []
    else:
        print("Loaded", len(data), "expense(s) from", DATA_FILE)
        return data
    finally:
        print("(finished loading)")             # finally always runs, with or without an error


def save_expenses(expenses):
    """Write every expense to the JSON file. Called after every change."""
    with open(DATA_FILE, "w", encoding="utf-8") as file:
        json.dump(expenses, file, indent=2)


def export_csv(expenses):
    """Write the expenses to a CSV file that a spreadsheet can open."""
    with open(CSV_FILE, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["date", "category", "description", "amount"])
        writer.writeheader()
        writer.writerows(expenses)
    print("Exported", len(expenses), "expense(s) to", CSV_FILE)


print("========================================")
print("   PERSONAL EXPENSE TRACKER")
print("========================================")
expenses = load_expenses()

while True:
    print()
    print("1) Add an expense   2) Show all   3) Totals by category   4) Filter by month   5) Export to CSV   6) Quit")
    choice = input("Choose 1-6: ").strip()

    if choice == "1":
        expense = add_expense(expenses)
        save_expenses(expenses)                  # save straight away
        print("✅ Saved:", expense["description"], f"({expense['amount']:.2f})")
    elif choice == "2":
        show_expenses(expenses)
    elif choice == "3":
        show_totals_by_category(expenses)
    elif choice == "4":
        month = input("Month (YYYY-MM): ").strip()
        chosen = [expense for expense in expenses if expense["date"].startswith(month)]
        show_expenses(chosen, "(no expenses in that month)")
    elif choice == "5":
        export_csv(expenses)
    elif choice == "6":
        print("Goodbye! Your expenses are saved in", DATA_FILE)
        break
    else:
        print("Please choose 1 to 6.")